In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
cus_predict = pd.read_csv(
    DATA_DIR / "customer_segmentation_prediction.csv",
    dtype={"fullVisitorId": str}
)

In [2]:
stars_df = cus_predict[cus_predict['cluster_label'] == 'Stars']

In [3]:
cus_predict.info()

<class 'pandas.DataFrame'>
RangeIndex: 119207 entries, 0 to 119206
Data columns (total 72 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   fullVisitorId                     119207 non-null  str    
 1   visitId_count                     119207 non-null  int64  
 2   visitNumber_max                   119207 non-null  int64  
 3   totals_hits_sum                   119207 non-null  int64  
 4   funnel_depth                      119207 non-null  float64
 5   totals_pageviews_sum              119207 non-null  float64
 6   totals_pageviews_mean             119207 non-null  float64
 7   totals_timeOnSite_sum             119207 non-null  float64
 8   totals_timeOnSite_mean            119207 non-null  float64
 9   totals_bounces_mean               119207 non-null  float64
 10  totals_sessionQualityDim_mean     119207 non-null  float64
 11  totals_sessionQualityDim_max      119207 non-null  int64  
 12 

In [4]:
N = len(stars_df)

In [ ]:
import pandas as pd
import numpy as np
import itertools

if "expected_revenue_usd" not in cus_predict.columns:
    cus_predict["expected_revenue_usd"] = (
        np.expm1(cus_predict["e_revenue"]) * cus_predict["p_buy"] / 1e6
    )
 
stars = cus_predict[cus_predict["cluster_label"] == "Stars"]
low   = cus_predict[cus_predict["cluster_label"] == "Low Priority"]

segments = {
    "Stars": {
        "N":         len(stars),
        "total_rev": stars["expected_revenue_usd"].sum(),  
        "p_buy_avg": stars["p_buy"].mean(),                
    },
    "Low Priority": {
        "N":         len(low),
        "total_rev": low["expected_revenue_usd"].sum(),
        "p_buy_avg": low["p_buy"].mean(),
    },
}
 


#   Expected Profit = total_rev × CR × GM − N × CAC
# Break-even RR = (N × CAC) / (total_rev × GM)
 
# Scenario
cac_values     = [0.01, 0.05, 0.20, 0.50]   # USD/user
response_rates = [0.02, 0.05, 0.10, 0.20]   # 2% → 20%
gross_margins  = [0.20, 0.30, 0.40, 0.50]   # 20% → 50%
 
def calc_profit(total_rev, N, rr, gm, cac):
    return total_rev * rr * gm - N * cac
 
def calc_roi(profit, N, cac):
    cost = N * cac
    return profit / cost if cost > 0 else np.nan
 
def calc_breakeven_rr(total_rev, N, gm, cac):
    denom = total_rev * gm
    return (N * cac) / denom if denom > 0 else np.inf
 
rows_sens = []
for seg_name, seg in segments.items():
    for cac, rr, gm in itertools.product(cac_values, response_rates, gross_margins):
        profit = calc_profit(seg["total_rev"], seg["N"], rr, gm, cac)
        be     = calc_breakeven_rr(seg["total_rev"], seg["N"], gm, cac)
        rows_sens.append({
            "segment":             seg_name,
            "N":                   seg["N"],
            "total_rev_usd":       round(seg["total_rev"], 2),
            "CAC_usd":             cac,
            "response_rate":       rr,
            "response_rate_pct":   rr * 100,
            "gross_margin":        gm,
            "gross_margin_pct":    gm * 100,
            "expected_profit_usd": round(profit, 2),
            "roi_pct":             round(calc_roi(profit, seg["N"], cac) * 100, 1),
            "breakeven_rr_pct":    round(min(be * 100, 9999), 2),
            "is_profitable":       profit > 0,
        })
 
df_sensitivity = pd.DataFrame(rows_sens)
 

In [ ]:
 
z_alpha_2        = 1.96   # two-tailed, alpha=0.05
z_beta           = 0.84   # power=0.80
MAX_REALISTIC_BE = 0.20   # BE-CR > 20% → unrealistic for campaign

BASELINE_RR = {
    "Stars":        0.05,
    "Low Priority": 0.02,
}

DAILY_TRAFFIC = {
    "Stars":        100,
    "Low Priority": 2_000,
}

rows_ab = []

for seg_name, grp in df_sensitivity.groupby("segment"):

    p1            = BASELINE_RR[seg_name]
    daily_traffic = DAILY_TRAFFIC[seg_name]
    N             = grp["N"].iloc[0]

    # Each (CAC, GM) combination has one break-even RR
    for (cac, gm), sub in grp.groupby(["CAC_usd", "gross_margin_pct"]):

        be_pct = sub["breakeven_rr_pct"].iloc[0]
        be     = be_pct / 100

        # Case 1: profitable at baseline already
        if be <= p1:

            rows_ab.append({
                "segment":              seg_name,
                "CAC_usd":              cac,
                "gross_margin_pct":     gm,
                "baseline_rr_pct":      round(p1 * 100, 2),
                "breakeven_rr_pct":     round(be_pct, 2),
                "mde_relative_pct":     None,
                "n_per_group":          None,
                "total_n_required":     None,
                "current_pool":         N,
                "days_to_run":          None,
                "feasibility":          "Profitable at baseline",
                "conclusion":           "Deploy immediately, no A/B test needed",
            })

            continue

        # Case 2: impossible break-even RR
        if be >= 1.0:

            rows_ab.append({
                "segment":              seg_name,
                "CAC_usd":              cac,
                "gross_margin_pct":     gm,
                "baseline_rr_pct":      round(p1 * 100, 2),
                "breakeven_rr_pct":     round(be_pct, 2),
                "mde_relative_pct":     None,
                "n_per_group":          None,
                "total_n_required":     None,
                "current_pool":         N,
                "days_to_run":          None,
                "feasibility":          "Infeasible",
                "conclusion":           "Break-even RR > 100%, impossible to achieve",
            })

            continue

        # Case 3: technically possible but unrealistic
        if be > MAX_REALISTIC_BE:

            rows_ab.append({
                "segment":              seg_name,
                "CAC_usd":              cac,
                "gross_margin_pct":     gm,
                "baseline_rr_pct":      round(p1 * 100, 2),
                "breakeven_rr_pct":     round(be_pct, 2),
                "mde_relative_pct":     None,
                "n_per_group":          None,
                "total_n_required":     None,
                "current_pool":         N,
                "days_to_run":          None,
                "feasibility":          "Unrealistic",
                "conclusion":           f"Break-even RR={be_pct:.1f}% exceeds realistic campaign benchmark (20%)",
            })

            continue

        # Case 4: feasible range → calculate sample size
        p2          = be
        mde_rel     = (p2 - p1) / p1

        n_per_group = int(np.ceil(
            ((z_alpha_2 + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)))
            / (p2 - p1) ** 2
        ))

        total_n = n_per_group * 2
        days    = int(np.ceil(total_n / daily_traffic))

        if total_n > N:

            feasibility = "Insufficient sample"
            conclusion  = f"Need {total_n:,} users but pool only has {N:,}"

        elif days <= 30:

            feasibility = "Feasible"
            conclusion  = f"Run for {days} days, requires {total_n:,} users"

        elif days <= 90:

            feasibility = "Long duration"
            conclusion  = f"Requires {days} days — consider cheaper scenarios"

        else:

            feasibility = "Infeasible"
            conclusion  = f"Requires {days} days — too long"

        rows_ab.append({
            "segment":              seg_name,
            "CAC_usd":              cac,
            "gross_margin_pct":     gm,
            "baseline_rr_pct":      round(p1 * 100, 2),
            "breakeven_rr_pct":     round(be_pct, 2),
            "mde_relative_pct":     round(mde_rel * 100, 1),
            "n_per_group":          n_per_group,
            "total_n_required":     total_n,
            "current_pool":         N,
            "days_to_run":          days,
            "feasibility":          feasibility,
            "conclusion":           conclusion,
        })

df_ab = pd.DataFrame(rows_ab)

In [7]:
# EXPORT CSV FOR DASHBOARD

df_sensitivity.to_csv("sensitivity_analysis.csv", index=False)
df_ab.to_csv("ab_test_feasibility.csv", index=False)

# ── Quick summary print

print("SEGMENT STATS")
print("=" * 55)

for seg_name, seg in segments.items():

    print(
        f"[{seg_name}]  "
        f"N={seg['N']:,}  |  "
        f"total_rev=${seg['total_rev']:,.2f}  |  "
        f"p_buy_avg={seg['p_buy_avg']:.2f}"
    )

print(f"\n{'=' * 65}")
print("SENSITIVITY — % OF PROFITABLE SCENARIOS")

pct = (
    df_sensitivity
    .groupby(["segment", "gross_margin_pct"])["is_profitable"]
    .mean()
    .mul(100)
    .round(1)
    .rename("profitable_%")
)

print(pct.to_string())

print(f"\n{'=' * 65}")
print("TOP 5 PROFITABLE SCENARIOS (HIGHEST ROI)")

top = (
    df_sensitivity[df_sensitivity["is_profitable"]]
    .sort_values("roi_pct", ascending=False)
    .groupby("segment")
    .head(5)[[
        "segment",
        "CAC_usd",
        "response_rate_pct",
        "gross_margin_pct",
        "expected_profit_usd",
        "roi_pct",
        "breakeven_rr_pct"
    ]]
)

print(top.to_string(index=False) if not top.empty else "  No profitable scenarios")

print(f"\n{'=' * 65}")
print("A/B TEST FEASIBILITY (TARGET = BREAK-EVEN RR)")
print("=" * 65)

print(
    df_ab[[
        "segment",
        "CAC_usd",
        "gross_margin_pct",
        "baseline_rr_pct",
        "breakeven_rr_pct",
        "mde_relative_pct",
        "total_n_required",
        "days_to_run",
        "feasibility",
        "conclusion"
    ]].to_string(index=False)
)

print(f"\n{'=' * 65}")
print("A/B TEST SUMMARY BY SEGMENT × FEASIBILITY")

print(
    df_ab.groupby(["segment", "feasibility"])
    .size()
    .rename("n_scenarios")
    .to_string()
)

SEGMENT STATS
[Stars]  N=2,462  |  total_rev=$5,964.25  |  p_buy_avg=0.04
[Low Priority]  N=116,745  |  total_rev=$5,881.00  |  p_buy_avg=0.00

SENSITIVITY — % OF PROFITABLE SCENARIOS
segment       gross_margin_pct
Low Priority  20.0                 0.0
              30.0                 0.0
              40.0                 0.0
              50.0                 0.0
Stars         20.0                25.0
              30.0                37.5
              40.0                37.5
              50.0                50.0

TOP 5 PROFITABLE SCENARIOS (HIGHEST ROI)
segment  CAC_usd  response_rate_pct  gross_margin_pct  expected_profit_usd  roi_pct  breakeven_rr_pct
  Stars     0.01               20.0              50.0               571.80   2322.5              0.83
  Stars     0.01               20.0              40.0               452.52   1838.0              1.03
  Stars     0.01               20.0              30.0               333.23   1353.5              1.38
  Stars     0.01       